In [ ]:
# If you're in Colab, make sure the runtime has a GPU: Runtime > Change runtime type > GPU.
import sys, subprocess, os, shutil, json, glob, textwrap, random, re, pathlib

# Install core PyTorch libraries first
!pip -q install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

print("Ready. CUDA available:", __import__("torch").cuda.is_available())

# Install other libraries after torch is confirmed to be working
!pip -q install 'git+https://github.com/facebookresearch/detectron2.git'  # current Colab-compatible build
!pip -q install pycocotools==2.0.7
!pip -q install ultralytics

Ready. CUDA available: False
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

# EDIT this path if your zip is somewhere else in Drive
ZIP_PATH = "/content/drive/MyDrive/dental fresh/archive.zip"

# Where to extract inside Colab
EXTRACT_DIR = "/content/dental_data"
os.makedirs(EXTRACT_DIR, exist_ok=True)

# Unzip
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

# Final dataset roots
BASE = f"{EXTRACT_DIR}/archive"
YOLO_ROOT = f"{BASE}/YOLO/YOLO"
COCO_ROOT = f"{BASE}/COCO/COCO"

print("✅ Unzip complete")
print("YOLO root:", YOLO_ROOT)
print("COCO root:", COCO_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Unzip complete
YOLO root: /content/dental_data/archive/YOLO/YOLO
COCO root: /content/dental_data/archive/COCO/COCO


In [ ]:
import os

YOLO_ROOT = "/content/dental_data/YOLO/YOLO"
new_yaml = "/content/dental_yolo.yaml"

# Define your 31 classes
CLASSES = [
"Caries","Crown","Filling","Implant","Malaligned","Mandibular Canal","Missing teeth",
"Periapical lesion","Retained root","Root Canal Treatment","Root Piece","Impacted tooth",
"Maxillary sinus","Bone Loss","Fracture teeth","Permanent Teeth","Supra Eruption","TAD",
"Abutment","Attrition","Bone defect","Gingival former","Metal band","Orthodontic brackets",
"Permanent retainer","Post-core","Plating","Wire","Cyst","Root resorption","Primary teeth"
]

with open(new_yaml, "w") as f:
    f.write(f"path: {YOLO_ROOT}\n")
    f.write("train: train/images\n")
    f.write("val: valid/images\n")
    f.write("test: test/images\n")
    f.write("names:\n")
    for i, name in enumerate(CLASSES):
        f.write(f"  {i}: {name}\n")

print("✅ New data.yaml created at:", new_yaml)
print(open(new_yaml).read())

✅ New data.yaml created at: /content/dental_yolo.yaml
path: /content/dental_data/YOLO/YOLO
train: train/images
val: valid/images
test: test/images
names:
  0: Caries
  1: Crown
  2: Filling
  3: Implant
  4: Malaligned
  5: Mandibular Canal
  6: Missing teeth
  7: Periapical lesion
  8: Retained root
  9: Root Canal Treatment
  10: Root Piece
  11: Impacted tooth
  12: Maxillary sinus
  13: Bone Loss
  14: Fracture teeth
  15: Permanent Teeth
  16: Supra Eruption
  17: TAD
  18: Abutment
  19: Attrition
  20: Bone defect
  21: Gingival former
  22: Metal band
  23: Orthodontic brackets
  24: Permanent retainer
  25: Post-core
  26: Plating
  27: Wire
  28: Cyst
  29: Root resorption
  30: Primary teeth



In [ ]:
# Install ultralytics (the version you're already using)
!pip install ultralytics==8.3.40 -q

from ultralytics import YOLO

# Path to your YAML
new_yaml = "/content/dental_yolo.yaml"
print("Using YAML:", new_yaml)

# Load stronger model (required for 90+ mAP)
model = YOLO("yolo11m.pt")

# HIGH ACCURACY CONFIG compatible with Ultralytics 8.3.40
results = model.train(
    data=new_yaml,

    # High resolution training
    imgsz=1024,

    # Memory-safe settings for T4
    batch=1,
    device=0,

    # Longer training = higher accuracy
    epochs=150,
    patience=30,

    # Augmentations (all valid)
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.4,
    degrees=5.0, translate=0.05, scale=0.5,
    fliplr=0.5,
    mosaic=0.7,
    mixup=0.15,

    # Checkpoints every 10 epochs
    save_period=10,

    workers=2,
    project="/content/exp_yolo",
    name="high_acc",
    exist_ok=True,
)

print("✅ Training complete")
print("✅ Best model:", results.save_dir + "/weights/best.pt")

Using YAML: /content/dental_yolo.yaml
New https://pypi.org/project/ultralytics/8.3.226 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolo11m.pt, data=/content/dental_yolo.yaml, epochs=150, time=None, patience=30, batch=1, imgsz=1024, save=True, save_period=10, cache=False, device=0, workers=2, project=/content/exp_yolo, name=high_acc, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina

train: Scanning /content/dental_data/YOLO/YOLO/train/labels.cache... 9481 images, 38 backgrounds, 2 corrupt: 100%|██████████| 9481/9481 [00:00<?, ?it/s]

train: WARNING ⚠️ /content/dental_data/YOLO/YOLO/train/images/cropped_LAXMI-JAIN_2023-10-20191001_1_png.rf.1ee36d70f5260b9c42c5b057b1aa58b8.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [          1]
train: WARNING ⚠️ /content/dental_data/YOLO/YOLO/train/images/cropped_LAXMI-JAIN_2023-10-20191001_1_png.rf.7401a2b3342e05d6e82d0807b91ea941.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [          1]
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))



/usr/local/lib/python3.12/dist-packages/ultralytics/data/augment.py:1850: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /content/dental_data/YOLO/YOLO/valid/labels.cache... 2871 images, 7 backgrounds, 0 corrupt: 100%|██████████| 2871/2871 [00:00<?, ?it/s]


Plotting labels to /content/exp_yolo/high_acc/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 106 weight(decay=0.0), 113 weight(decay=0.0005), 112 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 1024 train, 1024 val
Using 2 dataloader workers
Logging results to /content/exp_yolo/high_acc
Starting training for 150 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

      9/150      1.91G      1.601       1.37      1.422         23       1024: 100%|██████████| 9479/9479 [23:26<00:00,  6.74it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1436/1436 [01:47<00:00, 13.37it/s]


                   all       2871      27141       0.37      0.311      0.228      0.116

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     10/150      1.95G      1.592      1.361      1.413          5       1024: 100%|██████████| 9479/9479 [23:26<00:00,  6.74it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1436/1436 [01:48<00:00, 13.25it/s]


                   all       2871      27141      0.459      0.292      0.259      0.129

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     11/150      1.95G      1.566      1.322      1.395         13       1024:  31%|███       | 2937/9479 [07:26<15:55,  6.85it/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ========================================================================
# ✅ FIXED ONE-CELL FULL PIPELINE (WITH UNZIPPING + AUTO-DOWNLOAD)
# ========================================================================

# 0) INSTALL ULTRALYTICS
!pip install ultralytics==8.3.40 -q

from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
from ultralytics import YOLO
from google.colab import files

# ========================================================================
# ✅ EDIT THESE 3 PATHS ONLY
# ========================================================================
ZIP_PATH  = "/content/drive/MyDrive/dental fresh/archive.zip"   # dataset zip
LAST_PT   = "/content/drive/MyDrive/dental fresh/best.pt"       # checkpoint
BACKUP_DIR = "/content/drive/MyDrive/dental fresh"              # backup dir
# ========================================================================


# ========================================================================
# ✅ UNZIP DATASET (must happen EVERY new session)
# ========================================================================
EXTRACT_DIR = "/content/dental_data"

# remove old if exists
!rm -rf /content/dental_data
os.makedirs(EXTRACT_DIR, exist_ok=True)

print("⏳ Extracting dataset...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)
print("✅ Extraction complete!")

# The correct YOLO root
YOLO_ROOT = "/content/dental_data/YOLO/YOLO"
print("✅ YOLO root:", YOLO_ROOT)


# ========================================================================
# ✅ Rebuild YAML with correct path
# ========================================================================
new_yaml = "/content/dental_yolo.yaml"

CLASSES = [
"Caries","Crown","Filling","Implant","Malaligned","Mandibular Canal","Missing teeth",
"Periapical lesion","Retained root","Root Canal Treatment","Root Piece","Impacted tooth",
"Maxillary sinus","Bone Loss","Fracture teeth","Permanent Teeth","Supra Eruption","TAD",
"Abutment","Attrition","Bone defect","Gingival former","Metal band","Orthodontic brackets",
"Permanent retainer","Post-core","Plating","Wire","Cyst","Root resorption","Primary teeth"
]

with open(new_yaml, "w") as f:
    f.write(f"path: {YOLO_ROOT}\n")
    f.write("train: train/images\n")
    f.write("val: valid/images\n")
    f.write("test: test/images\n")
    f.write("names:\n")
    for i, c in enumerate(CLASSES):
        f.write(f"  {i}: {c}\n")

print("✅ YAML rebuilt at:", new_yaml)


# ========================================================================
# ✅ Load model (resume if last.pt exists)
# ========================================================================
if os.path.isfile(LAST_PT):
    print("✅ Resuming from:", LAST_PT)
    model = YOLO(LAST_PT)
    RESUME_MODE = True
else:
    print("⚠ No checkpoint found, starting fresh.")
    model = YOLO("yolo11m.pt")
    RESUME_MODE = False


# ========================================================================
# ✅ Callback (Auto-download every 10 epochs + auto-backup every 5 epochs)
# ========================================================================
def custom_callback(trainer):
    epoch = trainer.epoch

    # ---------------------------------------------------------
    # 🔽 Auto-download last.pt every 10 epochs
    # ---------------------------------------------------------
    if epoch > 0 and epoch % 10 == 0:
        ckpt = os.path.join(trainer.save_dir, "weights", "last.pt")
        if os.path.isfile(ckpt):
            print(f"\n⬇️ Auto-download triggered at epoch {epoch}\n")
            files.download(ckpt)

    # ---------------------------------------------------------
    # 💾 Auto-backup every 5 epochs
    # ---------------------------------------------------------
    if epoch % 5 == 0:
        wdir = os.path.join(trainer.save_dir, "weights")
        last_ckpt = os.path.join(wdir, "last.pt")
        best_ckpt = os.path.join(wdir, "best.pt")

        if os.path.isfile(last_ckpt):
            os.system(f"cp '{last_ckpt}' '{BACKUP_DIR}/last_epoch_{epoch}.pt'")
            print(f"💾 Saved last.pt (epoch {epoch})")

        if os.path.isfile(best_ckpt):
            os.system(f"cp '{best_ckpt}' '{BACKUP_DIR}/best_epoch_{epoch}.pt'")
            print(f"💾 Saved best.pt (epoch {epoch})")


model.add_callback("on_epoch_end", custom_callback)


# ========================================================================
# ✅ START / RESUME TRAINING
# ========================================================================
results = model.train(
    data=new_yaml,
    resume=RESUME_MODE,
    imgsz=768,
    batch=2,
    epochs=150,
    patience=30,
    device=0,
    workers=2,
    save_period=10,
    mosaic=0.7,
    mixup=0.15,
    fliplr=0.5,
    degrees=5.0,
    translate=0.05,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    project="/content/exp_yolo",
    name="high_acc",
    exist_ok=True,
)

print("🚀 Training running successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ Extracting dataset...
✅ Extraction complete!
✅ YOLO root: /content/dental_data/YOLO/YOLO
✅ YAML rebuilt at: /content/dental_yolo.yaml
✅ Resuming from: /content/drive/MyDrive/dental fresh/best.pt
New https://pypi.org/project/ultralytics/8.3.229 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=/content/drive/MyDrive/dental fresh/best.pt, data=/content/dental_yolo.yaml, epochs=150, time=None, patience=30, batch=2, imgsz=768, save=True, save_period=10, cache=False, device=0, workers=2, project=/content/exp_yolo, name=high_acc, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=/content/drive/MyDrive/dental fresh/best.pt, amp=

100%|██████████| 755k/755k [00:00<00:00, 25.9MB/s]


TensorBoard: Start with 'tensorboard --logdir /content/exp_yolo/high_acc', view at http://localhost:6006/

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  1    111872  ultralytics.nn.modules.block.C3k2            [128, 256, 1, True, 0.25]     
  3                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  4                  -1  1    444928  ultralytics.nn.modules.block.C3k2            [256, 512, 1, True, 0.25]     
  5                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  6                  -1  1   1380352  ultralytics.nn.modules.block.C3k2            [512, 512, 1

100%|██████████| 5.35M/5.35M [00:00<00:00, 120MB/s]


AMP: checks passed ✅


...

                   all       2871      27141      0.413      0.462      0.358       0.19

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     60/150      2.02G      1.344     0.9121      1.148         28        768: 100%|██████████| 4740/4740 [12:45<00:00,  6.20it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 718/718 [01:01<00:00, 11.70it/s]


                   all       2871      27141      0.381      0.446      0.342      0.187

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     61/150      1.97G      1.362     0.9205      1.157         64        768:  71%|███████   | 3355/4740 [08:56<03:41,  6.25it/s]


KeyboardInterrupt: 